<a href="https://colab.research.google.com/github/ms-starryvoid/ML_Lab_DataSet/blob/main/Programs/Exp4_HeartDiseaseDiagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas pgmpy scikit-learn

In [ ]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, BIC, BayesianEstimator
from pgmpy.inference import VariableElimination

data=pd.read_csv('https://raw.githubusercontent.com/ms-starryvoid/ML_Lab_DataSet/refs/heads/main/Lab_data/heart.csv')
print("Sample data set info ",data.head())
print(data.info())
# Discretize continuous variables using pandas cut()
def discretize(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels, include_lowest=True)

# Discretization bins and labels
data['age_cat'] = discretize(data['age'], bins=[28, 40, 55, 77], labels=['young', 'middle', 'old'])
data['trestbps_cat'] = discretize(data['trestbps'], bins=[94, 120, 140, 200], labels=['low', 'normal', 'high'])
data['chol_cat'] = discretize(data['chol'], bins=[126, 200, 300, 564], labels=['normal', 'high', 'very_high'])
data['thalach_cat'] = discretize(data['thalach'], bins=[71, 120, 160, 202], labels=['low', 'medium', 'high'])
data['oldpeak_cat'] = discretize(data['oldpeak'], bins=[-1, 1, 3, 6], labels=['low', 'medium', 'high'])

# Select discretized + categorical columns for model
model_data = data[['age_cat', 'sex', 'cp', 'trestbps_cat', 'chol_cat', 'fbs', 'restecg',
                   'thalach_cat', 'exang', 'oldpeak_cat', 'slope', 'ca', 'thal', 'target']]
 #Drop any rows with missing values (if any)
model_data = model_data.dropna()

# Learn structure with Hill Climbing and BIC score
hc = HillClimbSearch(model_data)
best_model = hc.estimate(scoring_method=BIC(model_data))


print("Learned network edges:")
print(best_model.edges())

# Create Bayesian Network model with learned structure
model = DiscreteBayesianNetwork(best_model.edges())
# Parameter estimation with Bayesian Estimator
model.fit(model_data, estimator=BayesianEstimator, prior_type='BDeu')

# Inference object for queries
inference = VariableElimination(model)

# Example query: Probability of heart disease (target=1) given some symptoms
evidence = {
    'age_cat': 'old',
    'sex': 1,
    'cp': 3,
    'trestbps_cat': 'high',
    'chol_cat': 'high',
    'fbs': 0,
    'exang': 1,
}

query_result = inference.query(variables=['target'], evidence=evidence)
print("\nProbability of Heart Disease given evidence:")
print(query_result)

evidence = {
    'age_cat': 'old',
    'sex': 1,
    'cp': 3,
    'trestbps_cat': 'high',
    'chol_cat': 'high',
    'fbs': 0,
    'restecg': 1,
    'thalach_cat': 'medium',
    'exang': 1,
    'oldpeak_cat': 'medium',
    'slope': 2,
    'ca': 0,
    'thal': 3,
}
query_result = inference.query(variables=['target'], evidence=evidence)
print(query_result)